# Silver Layer - Calendar

Transform raw Bronze calendar data into clean Silver data.

**Source:** `end-to-end_pipeline.bronze.calendar`  
**Target:** `end-to-end_pipeline.silver.calendar`

**Approach:** Profile → Inspect → Transform → Validate

## Step 1: Profile Bronze Data

**Inspect data quality issues before transformation:**

* Duplicate dates
* NULL values in key fields (date, year, month_number, day_name)
* Date range coverage (earliest to latest)
* Inconsistent day_name or quarter values

This single query checks all quality dimensions.

In [0]:
%sql

-- ============================================================
-- CELL 1: PROFILE BRONZE CALENDAR
-- Purpose: Identify duplicates, nulls, invalid dates,
--          and categorical inconsistencies
-- ============================================================

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT date) AS distinct_dates,
    COUNT(*) - COUNT(DISTINCT date) AS duplicate_dates,

    SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END)
        AS null_dates,

    SUM(CASE WHEN year IS NULL THEN 1 ELSE 0 END)
        AS null_years,

    SUM(CASE WHEN month_number IS NULL THEN 1 ELSE 0 END)
        AS null_month_numbers,

    SUM(CASE WHEN day_name IS NULL THEN 1 ELSE 0 END)
        AS null_day_names,

    COUNT(DISTINCT day_name)
        AS day_name_variations,

    COUNT(DISTINCT quarter)
        AS quarter_variations,

    MIN(date) AS earliest_date,
    MAX(date) AS latest_date

FROM `end-to-end_pipeline`.bronze.calendar;

## Step 2: Inspect Categorical Values

**Review actual day_name and quarter values to identify standardization needs:**

* Day name variations (Monday-Sunday)
* Quarter variations (Q1-Q4)
* Table schema review

Calendar dimension tables are typically clean, but we verify to be thorough.

In [0]:
%sql

-- ============================================================
-- CELL 2A: INSPECT DAY-OF-WEEK VALUES
-- ============================================================

SELECT
    day_name,
    COUNT(*) AS records
FROM `end-to-end_pipeline`.bronze.calendar
GROUP BY day_name
ORDER BY
    CASE day_name
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END;

In [0]:
%sql

-- ============================================================
-- CELL 2B: REVIEW CALENDAR SCHEMA
-- ============================================================

DESCRIBE TABLE `end-to-end_pipeline`.bronze.calendar;

## Step 3: Transform to Silver

**Calendar is a clean dimension table - minimal transformation needed:**

**Data Validation:**
* Ensure all dates are unique (already verified in profiling)
* Verify date range continuity
* Ensure data types are correct

**No standardization needed:**
* Day names are already properly formatted (Monday-Sunday)
* Quarters are already standardized (Q1-Q4)
* All numeric fields are correct data types

This creates a clean, analytics-ready Silver calendar dimension table.

In [0]:
%sql
-- ============================================================
-- STEP 3: TRANSFORM BRONZE → SILVER CALENDAR
-- ============================================================

CREATE OR REPLACE TABLE `end-to-end_pipeline`.silver.calendar AS

SELECT
    date,
    year,
    quarter,
    month_number,
    month_name,
    week_number,
    day_name,
    is_weekend

FROM `end-to-end_pipeline`.bronze.calendar

WHERE date IS NOT NULL

ORDER BY date;

## Step 4: Validate Silver Data

**Verify all transformations were successful.**

**Expected Results:**
* total_rows = 1,826 (5 years: 2021-2025)
* distinct_dates = 1,826
* remaining_duplicates = 0
* null_dates = 0
* day_name_count = 7 (Monday-Sunday)
* quarter_count = 4 (Q1-Q4)
* date_range = 2021-01-01 to 2025-12-31
* validation_status = 'PASS'

If any metric is unexpected, the transformation has an issue.

In [0]:
%sql

-- ============================================================
-- CELL 4: VALIDATE SILVER CALENDAR
-- Purpose: Validate uniqueness, completeness, date range,
--          and continuity of the Silver calendar table
-- ============================================================

WITH validation AS (

    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(*) - COUNT(DISTINCT date) AS remaining_duplicates,

        SUM(
            CASE
                WHEN date IS NULL THEN 1
                ELSE 0
            END
        ) AS null_dates,

        COUNT(DISTINCT day_name) AS day_name_count,
        COUNT(DISTINCT quarter) AS quarter_count,

        MIN(date) AS earliest_date,
        MAX(date) AS latest_date,

        DATEDIFF(MAX(date), MIN(date)) + 1 - COUNT(*) AS missing_dates

    FROM `end-to-end_pipeline`.silver.calendar
)

SELECT
    *,

    CASE
        WHEN total_rows = 1826
            AND distinct_dates = 1826
            AND remaining_duplicates = 0
            AND null_dates = 0
            AND day_name_count = 7
            AND quarter_count = 4
            AND missing_dates = 0
            AND earliest_date = DATE '2021-01-01'
            AND latest_date = DATE '2025-12-31'
        THEN 'PASS'
        ELSE 'FAIL'
    END AS validation_status

FROM validation;